# Fine-tune YOLOv8n-cls ngáp trên Kaggle GPU: gần và trung bình

Dataset là **native crop mặt từ runtime YuNet**, không phải ảnh chụp màn hình. Có 255 crop camera train, 320 ảnh static train và 160 ảnh static validation. Hai clip camera đều là train, cùng một người/buổi thu. Không có camera validation mới hoặc test độc lập trong gói này.

Chỉ Classify conv + BN affine + Linear được cập nhật; backbone và BN running buffers cố định. Downsampling/Gaussian/motion blur diễn ra **trước** RGB/resize/center crop 224×224. Validation giữ nguyên pixel/preprocessing runtime. Không dùng `YOLO.train()` mặc định vì transform mặc định khác quy trình dự án.

1. Tạo Kaggle Dataset **Private** từ `yawn_near_medium_dataset.zip`.
2. Import notebook này, Add Input dataset vừa tạo.
3. Settings → Accelerator → GPU (T4/P100 hoặc GPU đang có); bật Internet cho bước cài thư viện.
4. Run All. Cell GPU sẽ dừng ngay nếu chưa bật CUDA.

GPU tăng tốc huấn luyện; không thay thế dữ liệu đa góc/độc lập. Bản CPU thử trước đã học tốt hai clip train nhưng bỏ sót clip USB góc thấp cũ, nên checkpoint trong notebook tiếp tục mang nhãn **thử nghiệm**, chưa tự thay profile chính.

Nguồn hướng dẫn Kaggle: https://www.kaggle.com/docs/notebooks. Không gửi notebook/dataset lên Kaggle tự động.


In [ ]:
import os, sys, subprocess
from pathlib import Path
from importlib.metadata import version
assert (3, 9) <= sys.version_info[:2] < (3, 14), 'Profile hỗ trợ Python 3.9–3.13; cần kiểm tra wheel cho phiên bản mới.'
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/.config/ultralytics'
os.environ['MPLCONFIGDIR'] = '/kaggle/working/.config/matplotlib'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
requirements = Path('/kaggle/working/requirements-kaggle.lock.txt')
requirements.write_text('# Keep Kaggle\'s CUDA-enabled torch/torchvision; do not replace them with CPU wheels.\nultralytics==8.4.163\nonnx==1.17.0; python_version < "3.13"\nonnx==1.19.1; python_version >= "3.13"\nonnxruntime==1.20.1; python_version < "3.13"\nonnxruntime==1.23.2; python_version >= "3.13"\nnumpy==1.26.4; python_version < "3.13"\nnumpy==2.2.6; python_version >= "3.13"\nPillow==11.1.0\nopencv-python==4.11.0.86\n', encoding='utf-8')
# Chỉ đọc metadata: chưa import NumPy/torch trước khi cài dependency.
constraints = Path('/kaggle/working/gpu-stack-constraints.txt')
constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in ('torch', 'torchvision')), encoding='utf-8')
print('Python:', sys.version, '| CUDA stack:', constraints.read_text(), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    '--only-binary=:all:', '-c', str(constraints), '-r', str(requirements)])


## Kiểm tra GPU và tìm dataset (không cần sửa đường dẫn theo tên tài khoản)

In [ ]:
import torch, json, zipfile
assert torch.cuda.is_available(), 'Bật Settings > Accelerator > GPU rồi khởi động lại session.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

roots = list(Path('/kaggle/input').rglob('bundle.json'))
if not roots:
    archives = list(Path('/kaggle/input').rglob('yawn_near_medium_dataset.zip'))
    assert len(archives) == 1, 'Add Input dataset yawn_near_medium_dataset trước.'
    extracted = Path('/kaggle/working/native_dataset'); extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        assert all((extracted/n).resolve().is_relative_to(extracted.resolve()) for n in archive.namelist()), 'Unsafe ZIP path'
        archive.extractall(extracted)
    roots = list(extracted.rglob('bundle.json'))
roots = [p for p in roots if json.loads(p.read_text()).get('kind') == 'driver_monitor_yawn_near_medium']
assert len(roots) == 1, 'Giữ đúng một phiên bản dataset trong Input.'
DATA_ROOT = roots[0].parent
sys.path.insert(0, str(DATA_ROOT))
from kaggle_yawn_training import load_bundle, train
BUNDLE, DATA = load_bundle(DATA_ROOT)
print('Đã xác minh checksum và preprocessing:', DATA_ROOT)


## Kiểm tra nhãn, split và hình ảnh augmentation

In [ ]:
from collections import Counter
import cv2, matplotlib.pyplot as plt
from driver_monitor.yawn_distance import DistanceAugmentation, preview_views
print(Counter((r['domain'], r['split'], r['label']) for r in DATA['records']))
print('Class order:', DATA['runtime_spec']['classes'])
print('Camera validation: CHƯA CÓ; test độc lập: CHƯA CÓ')
recipe = DistanceAugmentation(**DATA['augmentation'])
sample = next(r for r in DATA['records'] if r['domain']=='camera' and r['distance']=='medium' and r['label']=='yawn')
native = cv2.imread(str(DATA_ROOT/sample['file']))
views = preview_views(native, DATA['runtime_spec'], recipe)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for axis, (name, rgb) in zip(axes.flat, views.items()):
    axis.imshow(rgb); axis.set_title(name); axis.axis('off')
fig.suptitle(f"Native ROI {native.shape[1]}×{native.shape[0]} → train augmentation → 224×224")
plt.tight_layout(); plt.show()


## Tham số fine-tune

Mặc định dùng bản USB refined (`usb_refined`) kế thừa model Kaggle gốc của bạn. Có thể đổi `START_FROM = 'original_kaggle'` để bắt đầu trực tiếp từ `camera_classical/best_yawn.pt` ban đầu. **Không dùng bản CPU candidate vừa train làm checkpoint khởi đầu.**

Batch 64 dùng GPU; giảm xuống 32/16 nếu hết VRAM. T4 ×2 chỉ dùng GPU đầu tiên ở bản này. Checkpoint chọn theo static validation, với guard không suy giảm F1/recall và ngáp nhầm so với chính checkpoint khởi đầu. Không dùng camera train để chọn weights. Ảnh cười/GENKI chỉ có lớp âm nên xem false-yawn, không diễn giải macro-F1 hai lớp riêng của chúng thành chất lượng toàn model.


In [ ]:
START_FROM = 'usb_refined'  # hoặc 'original_kaggle'
EPOCHS = 40
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
CAMERA_WEIGHT = 0.80
PATIENCE = 8
SEED = 42
NUM_WORKERS = 2
AMP = True
OUT = Path('/kaggle/working/yawn_kaggle_near_medium')
# Nếu chạy lại trong cùng session, đổi OUT sang tên mới; tool không xóa kết quả cũ.


## Train trên CUDA và kiểm tra bản lưu PT / xuất ONNX

In [ ]:
REPORT = train(DATA_ROOT, OUT, start_from=START_FROM, epochs=EPOCHS,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, camera_weight=CAMERA_WEIGHT,
    patience=PATIENCE, seed=SEED, workers=NUM_WORKERS, amp=AMP, require_gpu=True)
print('Checkpoint được chọn: epoch', REPORT['selected']['epoch'])
print('Trạng thái:', REPORT['status'])
print('PT ↔ ONNX:', json.loads((OUT/'parity.json').read_text()))


## Kết quả validation và biểu đồ loss

F1 ảnh/recall này là **validation ảnh cũ**. Không phải recall sự kiện, không phải kết quả test camera mới. Các clip lỗi đã khóa không được đưa vào bundle hoặc chọn checkpoint. Hãy kiểm tra video mới ở cả gần/trung bình, cười/nói chuyện và góc camera trước khi dùng profile mới.


In [ ]:
for source, metrics in REPORT['serialized_validation'].items():
    print(source, 'samples=', metrics['samples'], 'F1=', round(metrics['macro_f1'],4),
          'recall@0.70=', round(metrics['yawn_recall_at_070_including_unknown'],4),
          'false-yawn@0.70=', round(metrics['false_yawn_at_070'],4))
plt.figure(figsize=(9,3))
plt.plot([t['epoch'] for t in REPORT['trials']], [t['loss'] for t in REPORT['trials']])
plt.xlabel('Epoch'); plt.ylabel('Train loss'); plt.grid(alpha=.2); plt.show()


## Tải model về project Windows

Cell dưới tạo `yawn_kaggle_result.zip` trong Output. Tải ZIP, giải nén **vào thư mục `yolov10n/models`**, để có `models/yawn_kaggle_near_medium/runtime.yaml`. Model mắt/YuNet tham chiếu bộ hiện có trong project; không cần train lại chúng.

```powershell
& '.\.venv\Scripts\python.exe' -m driver_monitor run --config models/yawn_kaggle_near_medium/runtime.yaml --source 0 --backend onnx --display
```

PT/ONNX được tạo cho PC. Muốn dùng NPU Radxa cần chuyển RKNN trên Linux và đo riêng, chưa được nghiệm thu trong notebook này.


In [ ]:
import shutil
from IPython.display import FileLink, display
result_zip = shutil.make_archive('/kaggle/working/yawn_kaggle_result', 'zip',
    root_dir=OUT.parent, base_dir=OUT.name)
print(result_zip)
display(FileLink(result_zip))
